# NL2BI — Colab Text-to-SQL `/extract` service

End-to-end runner: mount Drive → load env → fetch repo → install deps → restore Postgres from Drive dump → load Qwen2.5-Coder-7B (4-bit) → boot FastAPI on `:8000` → expose via ngrok → smoke-test `/health` and `/extract`.

Run **Runtime → Run all** after attaching a GPU runtime (T4 / L4 / A100). Secrets (`NGROK_AUTHTOKEN`, `GITHUB_PAT`) live in `/content/drive/MyDrive/nl2bi_colab/.env`; never paste them into a cell.

In [2]:
# 1. Mount Google Drive (persists logs/artifacts and holds .env with secrets).
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

from pathlib import Path
DRIVE_BASE = Path('/content/drive/MyDrive/nl2bi_colab')
for sub in ('logs', 'artifacts', 'repo'):
    (DRIVE_BASE / sub).mkdir(parents=True, exist_ok=True)
print('Drive base:', DRIVE_BASE)

Mounted at /content/drive
Drive base: /content/drive/MyDrive/nl2bi_colab


In [3]:
!nvidia-smi

Mon May 11 19:08:43 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   31C    P0             47W /  600W |       0MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [4]:
# 2. Load env from Drive `.env` and set defaults. Tokens are never printed.
import os
from pathlib import Path

ENV_PATH = Path('/content/drive/MyDrive/nl2bi_colab/.env')
loaded = 0
if ENV_PATH.exists():
    for line in ENV_PATH.read_text(encoding='utf-8').splitlines():
        line = line.strip()
        if not line or line.startswith('#') or '=' not in line:
            continue
        key, value = line.split('=', 1)
        key, value = key.strip(), value.strip().strip('"').strip("'")
        os.environ[key] = value  # .env on Drive is the source of truth
        loaded += 1
    print(f'Loaded {loaded} keys from {ENV_PATH}')
else:
    print(f'No .env at {ENV_PATH} — falling back to existing env')

# Drive-fallback files for tokens we can write through the claude.ai Drive MCP
# (which can only create files at MyDrive root, not inside nested folders).
for env_var, file_paths in [
    ('COLAB_API_TOKEN', [
        '/content/drive/MyDrive/nl2bi_colab/.colab_api_token',
        '/content/drive/MyDrive/.colab_api_token',
    ]),
    ('BRIDGE_TOKEN', [
        '/content/drive/MyDrive/nl2bi_colab/.bridge_token',
        '/content/drive/MyDrive/.bridge_token',
    ]),
]:
    if not os.environ.get(env_var):
        for p in file_paths:
            try:
                text = Path(p).read_text(encoding='utf-8').strip()
            except (FileNotFoundError, OSError):
                continue
            if text:
                os.environ[env_var] = text
                break

os.environ.setdefault('COLAB_MODEL_ID', 'Qwen/Qwen2.5-Coder-7B-Instruct')
os.environ.setdefault('COLAB_QUANTIZATION', '4bit')
os.environ.setdefault('COLAB_MAX_NEW_TOKENS', '512')
os.environ.setdefault('COLAB_MOCK_MODEL', 'false')
os.environ.setdefault('COLAB_PORT', '8000')
os.environ.setdefault('COLAB_SPIDER_DB_ROOT', '/content/drive/MyDrive/diploma_plan_sql/data/spider/database')
os.environ.setdefault('COLAB_DATA_SOURCES_PATH', '/content/drive/MyDrive/nl2bi_colab/data_sources.pg.json')
os.environ.setdefault('COLAB_DEFAULT_DATA_SOURCE_ID', 'demo_concert_singer')
os.environ.setdefault('COLAB_ARTIFACTS_DIR', '/content/drive/MyDrive/nl2bi_colab/artifacts')
os.environ.setdefault('COLAB_LOG_DIR', '/content/drive/MyDrive/nl2bi_colab/logs')
# Auth + feature flags. Default to require_auth=true so any new deploy is
# locked down out of the box; downgrade only after explicit decision.
os.environ.setdefault('COLAB_REQUIRE_AUTH', 'true')
os.environ.setdefault('COLAB_DEBUG_ENDPOINTS', 'false')
os.environ.setdefault('COLAB_BRIDGE_ENABLED', 'false')

# Boolean presence only — token values stay in env / files.
for name in ('NGROK_AUTHTOKEN', 'GITHUB_PAT', 'COLAB_API_TOKEN', 'BRIDGE_TOKEN'):
    print(f'{name} present:', bool(os.environ.get(name)))
for name in ('COLAB_REQUIRE_AUTH', 'COLAB_DEBUG_ENDPOINTS', 'COLAB_BRIDGE_ENABLED'):
    print(f'{name}:', os.environ.get(name))


Loaded 16 keys from /content/drive/MyDrive/nl2bi_colab/.env
NGROK_AUTHTOKEN present: True
GITHUB_PAT present: True
COLAB_API_TOKEN present: True
BRIDGE_TOKEN present: True
COLAB_REQUIRE_AUTH: true
COLAB_DEBUG_ENDPOINTS: false
COLAB_BRIDGE_ENABLED: false


In [5]:
# 3. Clone (or update) the repo. Uses GITHUB_PAT from .env for the private repo.
import os, subprocess, shutil
from pathlib import Path
from urllib.parse import quote

GIT_OWNER = os.environ.get('NL2BI_GIT_OWNER', 'petrussia')
GIT_REPO = os.environ.get('NL2BI_GIT_REPO', 'NL2BI-AI-assistant')
# Default branch is `main` post-merge. Override via env if you want to test
# integration/* or a feature branch without re-editing the notebook.
GIT_BRANCH = os.environ.get('NL2BI_GIT_BRANCH', 'main')
REPO_DIR = Path('/content/nl2bi-colab')
TOKEN = os.environ.get('GITHUB_PAT')

if TOKEN:
    GIT_URL = f'https://x-access-token:{quote(TOKEN, safe="")}@github.com/{GIT_OWNER}/{GIT_REPO}.git'
else:
    GIT_URL = f'https://github.com/{GIT_OWNER}/{GIT_REPO}.git'

def _run(cmd, **kwargs):
    res = subprocess.run(cmd, capture_output=True, text=True, **kwargs)
    if res.returncode != 0:
        # Strip the token from any error output before printing.
        out = (res.stdout + res.stderr).replace(TOKEN or 'NO_TOKEN', '<token>')
        raise RuntimeError(f'`{" ".join(cmd)}` failed (rc={res.returncode}):\n{out}')
    return res

is_repo = (REPO_DIR / '.git').exists()
if REPO_DIR.exists() and not is_repo:
    # Stale partial dir from a previous failed clone: wipe it.
    print('Wiping stale', REPO_DIR)
    shutil.rmtree(REPO_DIR)
    is_repo = False

if is_repo:
    _run(['git', '-C', str(REPO_DIR), 'remote', 'set-url', 'origin', GIT_URL])
    _run(['git', '-C', str(REPO_DIR), 'fetch', '--all', '--prune'])
    _run(['git', '-C', str(REPO_DIR), 'checkout', GIT_BRANCH])
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], capture_output=True, text=True)
else:
    if not TOKEN:
        raise RuntimeError(
            'GITHUB_PAT missing. Add it to /content/drive/MyDrive/nl2bi_colab/.env\n'
            '  GITHUB_PAT=ghp_xxx_with_repo_read_scope\n'
            'then re-run cells 2 and 3.'
        )
    _run(['git', 'clone', '--branch', GIT_BRANCH, GIT_URL, str(REPO_DIR)])

# Make sure the token never lingers in the on-disk remote URL.
_run(['git', '-C', str(REPO_DIR), 'remote', 'set-url', 'origin',
      f'https://github.com/{GIT_OWNER}/{GIT_REPO}.git'])

print('Repo at:', REPO_DIR)
print('Branch:', GIT_BRANCH)
print('Tree top:', sorted(p.name for p in REPO_DIR.iterdir() if not p.name.startswith('.')))
head = subprocess.run(['git', '-C', str(REPO_DIR), 'log', '-1', '--oneline'], capture_output=True, text=True).stdout.strip()
print('HEAD:', head)


Repo at: /content/nl2bi-colab
Branch: main
Tree top: ['README.md', 'apps', 'artifacts', 'colab', 'contracts', 'demo_data', 'docs', 'pytest.ini', 'requirements.txt', 'scripts', 'services', 'tests']
HEAD: 071569f fix(web): keep model picker trigger short to avoid header wrap


In [6]:
# 4. Install pinned deps.
import subprocess, sys
REQ = '/content/nl2bi-colab/colab/requirements-colab.txt'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', REQ], check=True)
print('deps ok')

deps ok


In [ ]:
# 5. Restore local PostgreSQL from a dump stored on Google Drive.
# Idempotent: re-runs start Postgres if needed and skip restore when the same
# dump fingerprint is already recorded in the local database.
import fnmatch
import json
import os
import shlex
import shutil
import subprocess
import time
from pathlib import Path

REPO_DIR = Path(os.environ.get('COLAB_REPO_DIR', '/content/nl2bi-colab'))
os.environ['COLAB_REPO_DIR'] = str(REPO_DIR)
DRIVE_BASE = Path(os.environ.get('COLAB_DRIVE_BASE', '/content/drive/MyDrive/nl2bi_colab'))
MYDRIVE = Path('/content/drive/MyDrive')

PG_DB = os.environ.get('COLAB_PG_DB', 'northwind_ru')
PG_OWNER = os.environ.get('COLAB_PG_OWNER', 'nl2bi_owner')
PG_RO_USER = os.environ.get('COLAB_PG_RO_USER', 'nl2bi_ro')
PG_RO_PASSWORD = os.environ.get('COLAB_PG_RO_PASSWORD', 'nl2bi_ro')
PG_HOST = os.environ.get('COLAB_PG_HOST', '127.0.0.1')
PG_PORT = int(os.environ.get('COLAB_PG_PORT', '5432'))
PG_SOURCE_ID = os.environ.get('COLAB_PG_DATA_SOURCE_ID', PG_DB)
PG_SCHEMAS = [s.strip() for s in os.environ.get('COLAB_PG_SCHEMAS', 'public').split(',') if s.strip()]
PG_FORCE_RESTORE = os.environ.get('COLAB_PG_FORCE_RESTORE', '').strip().lower() in {'1', 'true', 'yes', 'y', 'on'}
PG_AUTO_BUILD_DUMP = os.environ.get('COLAB_PG_AUTO_BUILD_DUMP', 'true').strip().lower() in {'1', 'true', 'yes', 'y', 'on'}
PG_SET_DEFAULT = os.environ.get('COLAB_PG_SET_DEFAULT', '').strip().lower() in {'1', 'true', 'yes', 'y', 'on'}

_SECRET_VALUES = [
    PG_RO_PASSWORD,
    os.environ.get('NGROK_AUTHTOKEN'),
    os.environ.get('GITHUB_PAT'),
    os.environ.get('COLAB_API_TOKEN'),
    os.environ.get('BRIDGE_TOKEN'),
]


def _redact(text):
    text = text or ''
    for secret in _SECRET_VALUES:
        if secret:
            text = text.replace(secret, '<secret>')
    return text


def _run(cmd, *, input=None, env=None, check=True):
    cmd = [str(part) for part in cmd]
    res = subprocess.run(
        cmd,
        input=input,
        env=env,
        capture_output=True,
        text=True,
    )
    if check and res.returncode != 0:
        display = _redact(' '.join(shlex.quote(part) for part in cmd))
        stdout = _redact(res.stdout[-4000:])
        stderr = _redact(res.stderr[-4000:])
        raise RuntimeError(
            f'command failed rc={res.returncode}: {display}\n'
            f'--- stdout tail ---\n{stdout}\n'
            f'--- stderr tail ---\n{stderr}'
        )
    return res


def _sudo(cmd):
    cmd = [str(part) for part in cmd]
    if os.geteuid() == 0 or not shutil.which('sudo'):
        return cmd
    return ['sudo'] + cmd


def _as_postgres(cmd):
    cmd = [str(part) for part in cmd]
    if shutil.which('sudo'):
        return ['sudo', '-u', 'postgres'] + cmd
    if shutil.which('runuser'):
        return ['runuser', '-u', 'postgres', '--'] + cmd
    return cmd


def _sql_ident(value):
    return '"' + str(value).replace('"', '""') + '"'


def _sql_lit(value):
    return "'" + str(value).replace("'", "''") + "'"


def _psql_admin(sql, *, database='postgres', check=True):
    return _run(
        _as_postgres(['psql', '-v', 'ON_ERROR_STOP=1', '-q', '-d', database]),
        input=sql,
        check=check,
    )


def _psql_admin_scalar(sql, *, database='postgres', check=True):
    res = _run(
        _as_postgres(['psql', '-v', 'ON_ERROR_STOP=1', '-Atq', '-d', database]),
        input=sql,
        check=check,
    )
    if res.returncode != 0:
        return None
    return res.stdout.strip()


def _pg_ready():
    res = _run(
        ['pg_isready', '-h', PG_HOST, '-p', str(PG_PORT)],
        check=False,
    )
    return res.returncode == 0


def _ensure_postgres_packages():
    missing = [name for name in ('psql', 'pg_restore', 'pg_isready') if not shutil.which(name)]
    if not missing:
        print('PostgreSQL client/server tools already installed')
        return
    print('Installing PostgreSQL packages:', ', '.join(missing))
    env = os.environ.copy()
    env['DEBIAN_FRONTEND'] = 'noninteractive'
    _run(_sudo(['apt-get', 'update', '-qq']), env=env)
    _run(_sudo(['apt-get', 'install', '-y', '-qq', 'postgresql', 'postgresql-client']), env=env)


def _start_postgres():
    if _pg_ready():
        print(f'PostgreSQL already ready on {PG_HOST}:{PG_PORT}')
        return
    print('Starting PostgreSQL service')
    _run(_sudo(['service', 'postgresql', 'start']), check=False)
    deadline = time.time() + 60
    while time.time() < deadline:
        if _pg_ready():
            print(f'PostgreSQL ready on {PG_HOST}:{PG_PORT}')
            return
        time.sleep(2)

    clusters = _run(['pg_lsclusters', '--no-header'], check=False)
    for line in clusters.stdout.splitlines():
        parts = line.split()
        if len(parts) >= 2:
            _run(_sudo(['pg_ctlcluster', parts[0], parts[1], 'start']), check=False)
    deadline = time.time() + 60
    while time.time() < deadline:
        if _pg_ready():
            print(f'PostgreSQL ready on {PG_HOST}:{PG_PORT}')
            return
        time.sleep(2)
    raise RuntimeError('PostgreSQL did not become ready')


def _walk_matching_paths(root, patterns, *, max_depth=5):
    root = Path(root)
    if not root.exists() or not root.is_dir():
        return
    base_depth = len(root.parts)
    lowered_patterns = [pat.lower() for pat in patterns]
    for current, dirnames, filenames in os.walk(root):
        current_path = Path(current)
        depth = len(current_path.parts) - base_depth
        if depth >= max_depth:
            dirnames[:] = []
        else:
            dirnames[:] = [d for d in dirnames if d not in {'.git', '__pycache__', '.ipynb_checkpoints'}]
        for filename in filenames:
            lower = filename.lower()
            if any(fnmatch.fnmatch(lower, pat) for pat in lowered_patterns):
                yield current_path / filename


def _candidate_dump_paths():
    explicit = os.environ.get('COLAB_PG_DUMP_PATH')
    if explicit:
        yield Path(explicit).expanduser()
        return

    names = [
        f'{PG_DB}.dump',
        f'{PG_DB}.backup',
        f'{PG_DB}.tar',
        f'{PG_DB}.sql',
        f'{PG_DB}.sql.gz',
        'northwind_ru.dump',
        'northwind_ru.backup',
        'northwind_ru.sql',
        'northwind_ru.sql.gz',
    ]
    roots = [
        DRIVE_BASE / 'db_dumps',
        DRIVE_BASE / 'dumps',
        DRIVE_BASE,
        MYDRIVE / 'nl2bi_demo' / 'postgres',
        MYDRIVE / 'nl2bi_demo' / 'db_dumps',
        MYDRIVE / 'nl2bi_demo',
        MYDRIVE / 'diploma_plan_sql' / 'data' / 'postgres',
        MYDRIVE / 'diploma_plan_sql' / 'db_dumps',
        MYDRIVE / 'diploma_plan_sql',
    ]
    for root in roots:
        for name in names:
            yield root / name

    patterns = [
        f'*{PG_DB}*.dump',
        f'*{PG_DB}*.backup',
        f'*{PG_DB}*.sql',
        f'*{PG_DB}*.sql.gz',
        '*northwind*.dump',
        '*northwind*.backup',
        '*northwind*.sql',
        '*northwind*.sql.gz',
    ]
    for root in roots:
        yield from _walk_matching_paths(root, patterns, max_depth=5)


def _sql_value(value):
    if value is None:
        return 'NULL'
    if isinstance(value, bool):
        return 'TRUE' if value else 'FALSE'
    if isinstance(value, (int, float)):
        return str(value)
    return _sql_lit(value)


def _insert_rows_sql(table, columns, rows):
    quoted_columns = ', '.join(_sql_ident(c) for c in columns)
    values = []
    for row in rows:
        values.append('(' + ', '.join(_sql_value(v) for v in row) + ')')
    return f'INSERT INTO {_sql_ident(table)} ({quoted_columns}) VALUES\n' + ',\n'.join(values) + ';\n'


def _northwind_ru_seed_sql():
    regions = [
        (1, 'Москва', 'Центральный', 13.1),
        (2, 'Санкт-Петербург', 'Северо-Западный', 5.6),
        (3, 'Татарстан', 'Приволжский', 4.0),
        (4, 'Свердловская область', 'Уральский', 4.3),
        (5, 'Новосибирская область', 'Сибирский', 2.8),
        (6, 'Краснодарский край', 'Южный', 5.8),
        (7, 'Приморский край', 'Дальневосточный', 1.9),
        (8, 'Ростовская область', 'Южный', 4.2),
    ]
    clients = [
        (1, 'ООО Ромашка', 'Иван Петров', 'Москва', 1, 'B2B'),
        (2, 'Северная торговля', 'Анна Смирнова', 'Санкт-Петербург', 2, 'Розница'),
        (3, 'Казань Маркет', 'Марат Сафин', 'Казань', 3, 'HoReCa'),
        (4, 'Урал Снаб', 'Ольга Морозова', 'Екатеринбург', 4, 'B2B'),
        (5, 'Сибирь Ритейл', 'Павел Никитин', 'Новосибирск', 5, 'Розница'),
        (6, 'Кубань Продукт', 'Наталья Белова', 'Краснодар', 6, 'HoReCa'),
        (7, 'Владивосток Фуд', 'Игорь Ли', 'Владивосток', 7, 'HoReCa'),
        (8, 'Дон HoReCa', 'Сергей Ковалев', 'Ростов-на-Дону', 8, 'HoReCa'),
        (9, 'Столица Кафе', 'Марина Зайцева', 'Москва', 1, 'HoReCa'),
        (10, 'Волга Опт', 'Алия Хасанова', 'Казань', 3, 'B2B'),
    ]
    employees = [
        (1, 'Мария Орлова', 'Менеджер по продажам', '2020-01-15', 1),
        (2, 'Дмитрий Волков', 'Старший менеджер', '2019-06-10', 2),
        (3, 'Елена Кузнецова', 'Менеджер по HoReCa', '2021-03-22', 3),
        (4, 'Алексей Соколов', 'Руководитель продаж', '2018-11-05', 4),
    ]
    categories = [
        (1, 'Напитки', 'Чай, кофе, соки и безалкогольные напитки'),
        (2, 'Кондитерские изделия', 'Шоколад, печенье и сладости'),
        (3, 'Молочные продукты', 'Сыры, йогурты и масло'),
        (4, 'Бакалея', 'Крупы, макароны и товары длительного хранения'),
        (5, 'Морепродукты', 'Охлажденная и замороженная рыба, креветки'),
    ]
    products = [
        (1, 'Чай черный', 1, 650, 'короб 10 кг', False),
        (2, 'Кофе зерновой', 1, 1450, 'короб 10 кг', False),
        (3, 'Шоколад темный', 2, 240, 'короб 24 шт', False),
        (4, 'Печенье овсяное', 2, 180, 'короб 30 шт', False),
        (5, 'Сыр выдержанный', 3, 820, 'кг', False),
        (6, 'Йогурт натуральный', 3, 90, 'упаковка', False),
        (7, 'Рис жасмин', 4, 160, 'кг', False),
        (8, 'Макароны premium', 4, 140, 'кг', False),
        (9, 'Лосось охлажденный', 5, 2100, 'кг', False),
        (10, 'Креветки северные', 5, 1750, 'кг', False),
        (11, 'Сок яблочный', 1, 120, 'упаковка', False),
        (12, 'Масло сливочное', 3, 260, 'кг', False),
    ]
    orders = [
        (101, 1, 1, '2024-01-10', '2024-01-13', 850),
        (102, 3, 3, '2024-01-18', '2024-01-21', 1300),
        (103, 2, 2, '2024-02-05', '2024-02-08', 900),
        (104, 6, 1, '2024-02-20', '2024-02-23', 1200),
        (105, 4, 4, '2024-03-03', '2024-03-07', 1100),
        (106, 5, 2, '2024-03-14', '2024-03-18', 700),
        (107, 1, 1, '2024-04-02', '2024-04-05', 1500),
        (108, 7, 3, '2024-04-15', '2024-04-19', 2100),
        (109, 8, 4, '2024-05-06', '2024-05-09', 950),
        (110, 9, 1, '2024-05-24', '2024-05-27', 800),
        (111, 10, 3, '2024-06-12', '2024-06-15', 1250),
        (112, 2, 2, '2024-07-08', '2024-07-11', 780),
        (113, 6, 1, '2024-08-21', '2024-08-25', 1800),
        (114, 3, 3, '2024-09-09', '2024-09-12', 990),
        (115, 4, 4, '2024-10-18', '2024-10-22', 1350),
        (116, 5, 2, '2024-11-11', '2024-11-14', 740),
        (117, 7, 3, '2024-12-03', '2024-12-07', 2200),
        (118, 9, 1, '2024-12-19', '2024-12-22', 1000),
    ]
    order_items = [
        (1, 101, 2, 1450, 300, 0.00), (2, 101, 1, 650, 220, 0.00), (3, 101, 3, 240, 400, 0.00),
        (4, 102, 9, 2100, 180, 0.05), (5, 102, 10, 1750, 160, 0.00),
        (6, 103, 3, 240, 900, 0.00), (7, 103, 4, 180, 1200, 0.00), (8, 103, 11, 120, 1500, 0.00),
        (9, 104, 2, 1450, 250, 0.00), (10, 104, 5, 820, 300, 0.00),
        (11, 105, 7, 160, 2400, 0.00), (12, 105, 8, 140, 3000, 0.00),
        (13, 106, 6, 90, 2500, 0.00), (14, 106, 12, 260, 900, 0.00),
        (15, 107, 2, 1450, 600, 0.07), (16, 107, 1, 650, 500, 0.00),
        (17, 108, 9, 2100, 300, 0.00), (18, 108, 10, 1750, 220, 0.00),
        (19, 109, 11, 120, 5000, 0.00), (20, 109, 4, 180, 1500, 0.00),
        (21, 110, 3, 240, 1000, 0.00), (22, 110, 2, 1450, 200, 0.00),
        (23, 111, 7, 160, 4000, 0.00), (24, 111, 8, 140, 3500, 0.00),
        (25, 112, 5, 820, 700, 0.00), (26, 112, 6, 90, 3000, 0.00),
        (27, 113, 10, 1750, 300, 0.00), (28, 113, 9, 2100, 250, 0.00),
        (29, 114, 2, 1450, 450, 0.00), (30, 114, 1, 650, 600, 0.00),
        (31, 115, 8, 140, 5000, 0.00), (32, 115, 7, 160, 3000, 0.00),
        (33, 116, 3, 240, 1800, 0.00), (34, 116, 4, 180, 2400, 0.00),
        (35, 117, 9, 2100, 400, 0.00), (36, 117, 10, 1750, 200, 0.00),
        (37, 118, 11, 120, 4000, 0.00), (38, 118, 2, 1450, 350, 0.00),
    ]
    ddl = '''
SET client_encoding = 'UTF8';
DROP TABLE IF EXISTS "позиции_заказа" CASCADE;
DROP TABLE IF EXISTS "заказы" CASCADE;
DROP TABLE IF EXISTS "товары" CASCADE;
DROP TABLE IF EXISTS "категории" CASCADE;
DROP TABLE IF EXISTS "клиенты" CASCADE;
DROP TABLE IF EXISTS "сотрудники" CASCADE;
DROP TABLE IF EXISTS "регионы" CASCADE;

CREATE TABLE "регионы" (
  "регион_id" integer PRIMARY KEY,
  "название" text NOT NULL,
  "федеральный_округ" text NOT NULL,
  "население_млн" real NOT NULL
);
CREATE TABLE "клиенты" (
  "клиент_id" integer PRIMARY KEY,
  "название_компании" text NOT NULL,
  "контактное_лицо" text NOT NULL,
  "город" text NOT NULL,
  "регион_id" integer NOT NULL REFERENCES "регионы"("регион_id"),
  "сегмент" text NOT NULL
);
CREATE TABLE "сотрудники" (
  "сотрудник_id" integer PRIMARY KEY,
  "фио" text NOT NULL,
  "должность" text NOT NULL,
  "дата_найма" date NOT NULL,
  "регион_id" integer NOT NULL REFERENCES "регионы"("регион_id")
);
CREATE TABLE "категории" (
  "категория_id" integer PRIMARY KEY,
  "название" text NOT NULL,
  "описание" text NOT NULL
);
CREATE TABLE "товары" (
  "товар_id" integer PRIMARY KEY,
  "название" text NOT NULL,
  "категория_id" integer NOT NULL REFERENCES "категории"("категория_id"),
  "цена_за_единицу" numeric(12,2) NOT NULL,
  "единица_измерения" text NOT NULL,
  "снят_с_продажи" boolean NOT NULL DEFAULT false
);
CREATE TABLE "заказы" (
  "заказ_id" integer PRIMARY KEY,
  "клиент_id" integer NOT NULL REFERENCES "клиенты"("клиент_id"),
  "сотрудник_id" integer NOT NULL REFERENCES "сотрудники"("сотрудник_id"),
  "дата_заказа" date NOT NULL,
  "дата_доставки" date,
  "стоимость_доставки" numeric(12,2) NOT NULL DEFAULT 0
);
CREATE TABLE "позиции_заказа" (
  "позиция_id" integer PRIMARY KEY,
  "заказ_id" integer NOT NULL REFERENCES "заказы"("заказ_id") ON DELETE CASCADE,
  "товар_id" integer NOT NULL REFERENCES "товары"("товар_id"),
  "цена_за_единицу" numeric(12,2) NOT NULL,
  "количество" integer NOT NULL CHECK ("количество" > 0),
  "скидка" numeric(5,2) NOT NULL DEFAULT 0 CHECK ("скидка" >= 0 AND "скидка" < 1)
);
'''
    inserts = [
        _insert_rows_sql('регионы', ['регион_id', 'название', 'федеральный_округ', 'население_млн'], regions),
        _insert_rows_sql('клиенты', ['клиент_id', 'название_компании', 'контактное_лицо', 'город', 'регион_id', 'сегмент'], clients),
        _insert_rows_sql('сотрудники', ['сотрудник_id', 'фио', 'должность', 'дата_найма', 'регион_id'], employees),
        _insert_rows_sql('категории', ['категория_id', 'название', 'описание'], categories),
        _insert_rows_sql('товары', ['товар_id', 'название', 'категория_id', 'цена_за_единицу', 'единица_измерения', 'снят_с_продажи'], products),
        _insert_rows_sql('заказы', ['заказ_id', 'клиент_id', 'сотрудник_id', 'дата_заказа', 'дата_доставки', 'стоимость_доставки'], orders),
        _insert_rows_sql('позиции_заказа', ['позиция_id', 'заказ_id', 'товар_id', 'цена_за_единицу', 'количество', 'скидка'], order_items),
    ]
    indexes = '''
CREATE INDEX "idx_клиенты_регион" ON "клиенты"("регион_id");
CREATE INDEX "idx_заказы_клиент" ON "заказы"("клиент_id");
CREATE INDEX "idx_заказы_сотрудник" ON "заказы"("сотрудник_id");
CREATE INDEX "idx_заказы_дата" ON "заказы"("дата_заказа");
CREATE INDEX "idx_позиции_заказа_заказ" ON "позиции_заказа"("заказ_id");
CREATE INDEX "idx_позиции_заказа_товар" ON "позиции_заказа"("товар_id");
ANALYZE;
'''
    return ddl + '\n'.join(inserts) + indexes


def _build_seed_dump_from_code():
    if PG_DB != 'northwind_ru':
        return None
    out_path = Path(os.environ.get('COLAB_PG_BUILD_DUMP_PATH', str(DRIVE_BASE / 'db_dumps' / f'{PG_DB}.dump')))
    out_path.parent.mkdir(parents=True, exist_ok=True)
    tmp_db = f'{PG_DB}_seed_tmp'
    tmp_dir = Path('/tmp/nl2bi_pg_seed')
    tmp_dir.mkdir(parents=True, exist_ok=True)
    tmp_dir.chmod(0o777)
    tmp_dump = tmp_dir / f'{PG_DB}.dump'
    if tmp_dump.exists():
        tmp_dump.unlink()
    print('No PostgreSQL dump found; building seed dump from bundled Northwind RU fixture')
    try:
        _psql_admin(
            f'''
SELECT pg_terminate_backend(pid)
FROM pg_stat_activity
WHERE datname = {_sql_lit(tmp_db)} AND pid <> pg_backend_pid();
DROP DATABASE IF EXISTS {_sql_ident(tmp_db)};
CREATE DATABASE {_sql_ident(tmp_db)} OWNER {_sql_ident(PG_OWNER)};
'''
        )
        _psql_admin(_northwind_ru_seed_sql(), database=tmp_db)
        _run(_as_postgres(['pg_dump', '--format=custom', '--no-owner', '--no-acl', '--file', str(tmp_dump), tmp_db]))
        shutil.copy2(tmp_dump, out_path)
        out_path.chmod(0o644)
        print('Built PostgreSQL dump:', out_path)
        return out_path
    finally:
        _psql_admin(
            f'''
SELECT pg_terminate_backend(pid)
FROM pg_stat_activity
WHERE datname = {_sql_lit(tmp_db)} AND pid <> pg_backend_pid();
DROP DATABASE IF EXISTS {_sql_ident(tmp_db)};
''',
            check=False,
        )


def _resolve_dump_path():
    seen = set()
    existing = []
    for path in _candidate_dump_paths():
        path = path.expanduser()
        key = str(path)
        if key in seen:
            continue
        seen.add(key)
        if path.is_file():
            existing.append(path)
    if not existing and PG_AUTO_BUILD_DUMP:
        built = _build_seed_dump_from_code()
        if built and built.is_file():
            existing.append(built)
    if not existing:
        checked_roots = [
            str(DRIVE_BASE),
            str(MYDRIVE / 'nl2bi_demo'),
            str(MYDRIVE / 'diploma_plan_sql'),
        ]
        raise RuntimeError(
            'PostgreSQL dump not found on Drive and auto-build did not produce one. '
            'Put northwind_ru.dump/.backup/.sql/.sql.gz under nl2bi_colab, nl2bi_demo, '
            'or diploma_plan_sql; or set COLAB_PG_DUMP_PATH=/content/drive/MyDrive/... '
            'in .env. Checked roots: ' + ', '.join(checked_roots)
        )
    existing.sort(key=lambda p: p.stat().st_mtime, reverse=True)
    if len(existing) > 1:
        print('PG dump candidates found:')
        for item in existing[:10]:
            print(' -', item)
        print('Using newest:', existing[0])
    return existing[0]


def _dump_fingerprint(path):
    stat = path.stat()
    return json.dumps(
        {
            'path': str(path),
            'size': stat.st_size,
            'mtime_ns': stat.st_mtime_ns,
        },
        sort_keys=True,
        separators=(',', ':'),
    )


def _ensure_roles():
    _psql_admin(
        f"""
DO $$
BEGIN
  IF NOT EXISTS (SELECT 1 FROM pg_roles WHERE rolname = {_sql_lit(PG_OWNER)}) THEN
    EXECUTE format('CREATE ROLE %I LOGIN', {_sql_lit(PG_OWNER)});
  END IF;
  IF NOT EXISTS (SELECT 1 FROM pg_roles WHERE rolname = {_sql_lit(PG_RO_USER)}) THEN
    EXECUTE format('CREATE ROLE %I LOGIN', {_sql_lit(PG_RO_USER)});
  END IF;
  EXECUTE format('ALTER ROLE %I LOGIN PASSWORD %L', {_sql_lit(PG_RO_USER)}, {_sql_lit(PG_RO_PASSWORD)});
END $$;
"""
    )


def _database_exists():
    out = _psql_admin_scalar(f'SELECT 1 FROM pg_database WHERE datname = {_sql_lit(PG_DB)};')
    return out == '1'


def _ensure_database():
    if _database_exists():
        _psql_admin(f'ALTER DATABASE {_sql_ident(PG_DB)} OWNER TO {_sql_ident(PG_OWNER)};', check=False)
        return
    _psql_admin(f'CREATE DATABASE {_sql_ident(PG_DB)} OWNER {_sql_ident(PG_OWNER)};')


def _drop_recreate_database():
    _psql_admin(
        f"""
SELECT pg_terminate_backend(pid)
FROM pg_stat_activity
WHERE datname = {_sql_lit(PG_DB)} AND pid <> pg_backend_pid();
DROP DATABASE IF EXISTS {_sql_ident(PG_DB)};
CREATE DATABASE {_sql_ident(PG_DB)} OWNER {_sql_ident(PG_OWNER)};
"""
    )


def _user_table_count():
    out = _psql_admin_scalar(
        """
SELECT count(*)
FROM information_schema.tables
WHERE table_schema NOT IN ('pg_catalog', 'information_schema')
  AND table_name <> '__nl2bi_restore_meta';
""",
        database=PG_DB,
        check=False,
    )
    try:
        return int(out or '0')
    except ValueError:
        return 0


def _current_restore_fingerprint():
    return _psql_admin_scalar(
        """
SELECT dump_fingerprint
FROM public.__nl2bi_restore_meta
ORDER BY restored_at DESC
LIMIT 1;
""",
        database=PG_DB,
        check=False,
    )


def _copy_dump_to_tmp(dump_path):
    work_dir = Path('/tmp/nl2bi_pg_restore')
    work_dir.mkdir(parents=True, exist_ok=True)
    tmp_path = work_dir / dump_path.name
    needs_copy = True
    if tmp_path.exists():
        src = dump_path.stat()
        dst = tmp_path.stat()
        needs_copy = src.st_size != dst.st_size or int(src.st_mtime) != int(dst.st_mtime)
    if needs_copy:
        print('Copying dump from Drive to local tmp for postgres user access')
        shutil.copy2(dump_path, tmp_path)
        tmp_path.chmod(0o644)
    return tmp_path


def _restore_dump(dump_path):
    tmp_dump = _copy_dump_to_tmp(dump_path)
    _drop_recreate_database()
    lower = tmp_dump.name.lower()
    print('Restoring PostgreSQL dump:', dump_path)
    if lower.endswith('.sql.gz'):
        command = f'gzip -dc {shlex.quote(str(tmp_dump))} | psql -v ON_ERROR_STOP=1 -q -d {shlex.quote(PG_DB)}'
        _run(_as_postgres(['bash', '-lc', command]))
    elif lower.endswith('.sql'):
        _run(_as_postgres(['psql', '-v', 'ON_ERROR_STOP=1', '-q', '-d', PG_DB, '-f', str(tmp_dump)]))
    else:
        _run(
            _as_postgres([
                'pg_restore',
                '--clean',
                '--if-exists',
                '--no-owner',
                '--no-acl',
                '--exit-on-error',
                '-d',
                PG_DB,
                str(tmp_dump),
            ])
        )


def _record_restore(fingerprint):
    _psql_admin(
        f"""
CREATE SCHEMA IF NOT EXISTS public;
CREATE TABLE IF NOT EXISTS public.__nl2bi_restore_meta (
  dump_fingerprint text PRIMARY KEY,
  restored_at timestamptz NOT NULL DEFAULT now()
);
TRUNCATE public.__nl2bi_restore_meta;
INSERT INTO public.__nl2bi_restore_meta (dump_fingerprint) VALUES ({_sql_lit(fingerprint)});
""",
        database=PG_DB,
    )


def _grant_readonly():
    statements = [
        f'GRANT CONNECT ON DATABASE {_sql_ident(PG_DB)} TO {_sql_ident(PG_RO_USER)};',
        f'ALTER ROLE {_sql_ident(PG_RO_USER)} IN DATABASE {_sql_ident(PG_DB)} SET search_path = {", ".join(_sql_ident(s) for s in PG_SCHEMAS)};',
    ]
    for schema in PG_SCHEMAS:
        statements.extend([
            f'CREATE SCHEMA IF NOT EXISTS {_sql_ident(schema)};',
            f'GRANT USAGE ON SCHEMA {_sql_ident(schema)} TO {_sql_ident(PG_RO_USER)};',
            f'GRANT SELECT ON ALL TABLES IN SCHEMA {_sql_ident(schema)} TO {_sql_ident(PG_RO_USER)};',
            f'GRANT SELECT ON ALL SEQUENCES IN SCHEMA {_sql_ident(schema)} TO {_sql_ident(PG_RO_USER)};',
            f'ALTER DEFAULT PRIVILEGES IN SCHEMA {_sql_ident(schema)} GRANT SELECT ON TABLES TO {_sql_ident(PG_RO_USER)};',
            f'ALTER DEFAULT PRIVILEGES IN SCHEMA {_sql_ident(schema)} GRANT SELECT ON SEQUENCES TO {_sql_ident(PG_RO_USER)};',
        ])
    _psql_admin('\n'.join(statements), database=PG_DB)


def _write_data_sources_file():
    repo_sources = REPO_DIR / 'demo_data' / 'data_sources.json'
    configured = Path(os.environ.get('COLAB_DATA_SOURCES_PATH', str(repo_sources)))
    source_path = configured if configured.exists() else repo_sources
    if source_path.exists():
        sources = json.loads(source_path.read_text(encoding='utf-8'))
    else:
        sources = {}

    existing = sources.get(PG_SOURCE_ID, {}) if isinstance(sources.get(PG_SOURCE_ID), dict) else {}
    source_name = os.environ.get('COLAB_PG_DATA_SOURCE_NAME') or existing.get('name') or 'Northwind RU - sales (PostgreSQL)'
    schema_version = os.environ.get('COLAB_PG_SCHEMA_VERSION') or existing.get('schema_version') or 'v1'
    dsn = f'host={PG_HOST} port={PG_PORT} dbname={PG_DB} user={PG_RO_USER} password={PG_RO_PASSWORD}'
    sources[PG_SOURCE_ID] = {
        'engine': 'postgres',
        'dsn': dsn,
        'db_name': PG_DB,
        'name': source_name,
        'schema_version': schema_version,
        'pg_schemas': PG_SCHEMAS,
    }

    out_path = Path(os.environ.get('COLAB_PG_DATA_SOURCES_PATH', str(DRIVE_BASE / 'data_sources.pg.json')))
    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_path.write_text(json.dumps(sources, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
    os.environ['COLAB_DATA_SOURCES_PATH'] = str(out_path)
    if PG_SET_DEFAULT:
        os.environ['COLAB_DEFAULT_DATA_SOURCE_ID'] = PG_SOURCE_ID
    return out_path


def _readonly_probe():
    env = os.environ.copy()
    env['PGPASSWORD'] = PG_RO_PASSWORD
    sql = """
SELECT current_database() || ' as ' || current_user || ', user_tables=' || count(*)
FROM information_schema.tables
WHERE table_schema NOT IN ('pg_catalog', 'information_schema')
  AND table_name <> '__nl2bi_restore_meta';
"""
    res = _run(
        [
            'psql',
            '-h',
            PG_HOST,
            '-p',
            str(PG_PORT),
            '-U',
            PG_RO_USER,
            '-d',
            PG_DB,
            '-Atq',
            '-v',
            'ON_ERROR_STOP=1',
        ],
        input=sql,
        env=env,
    )
    return res.stdout.strip()


_ensure_postgres_packages()
_start_postgres()
_ensure_roles()
_ensure_database()

dump_path = _resolve_dump_path()
fingerprint = _dump_fingerprint(dump_path)
current_fingerprint = _current_restore_fingerprint()
table_count = _user_table_count()

if current_fingerprint == fingerprint and not PG_FORCE_RESTORE:
    print('PostgreSQL restore skipped: same dump fingerprint already recorded')
else:
    if current_fingerprint is None and table_count > 0 and not PG_FORCE_RESTORE:
        print(
            f'PostgreSQL database {PG_DB!r} has {table_count} user tables but no restore metadata; '
            'recreating it from the Drive dump to make this notebook state reproducible.'
        )
    _restore_dump(dump_path)
    _record_restore(fingerprint)

_grant_readonly()
data_sources_path = _write_data_sources_file()
probe = _readonly_probe()

print('PG_READY:', probe)
print('PG_DUMP:', dump_path)
print('PG_DATA_SOURCE_ID:', PG_SOURCE_ID)
print('COLAB_DATA_SOURCES_PATH:', data_sources_path)
print('COLAB_DEFAULT_DATA_SOURCE_ID:', os.environ.get('COLAB_DEFAULT_DATA_SOURCE_ID'))


In [7]:
# 6. GPU sanity check.
import sys
if '/content/nl2bi-colab' not in sys.path:
    sys.path.insert(0, '/content/nl2bi-colab')
from colab.gpu import gpu_info
import json
print(json.dumps(gpu_info(), ensure_ascii=False, indent=2))

{
  "device": "cuda",
  "gpu_name": "NVIDIA RTX PRO 6000 Blackwell Server Edition",
  "vram_total_gb": 94.97,
  "vram_free_gb": 94.43,
  "cuda_available": true
}


In [8]:
# 7. Launch uvicorn in background. Idempotent: kills any prior server on the port first.
import os, subprocess, time
from pathlib import Path
from urllib import request, error
import json as _json

REPO_DIR = '/content/nl2bi-colab'
PORT = int(os.environ.get('COLAB_PORT', '8000'))
LOG_DIR = Path(os.environ['COLAB_LOG_DIR'])
LOG_DIR.mkdir(parents=True, exist_ok=True)
STDOUT_PATH = LOG_DIR / 'uvicorn.stdout.log'
STDERR_PATH = LOG_DIR / 'uvicorn.stderr.log'

def _env_true(name):
    return os.environ.get(name, '').strip().lower() in {'1', 'true', 'yes', 'y', 'on'}

def _tail(path, lines=80):
    try:
        text = Path(path).read_text(encoding='utf-8', errors='replace')
    except FileNotFoundError:
        return f'<missing: {path}>'
    return '\n'.join(text.splitlines()[-lines:])

def _kill_port(port):
    """Kill any process bound to the given TCP port (so re-running this cell works)."""
    killed = False
    try:
        out = subprocess.run(['fuser', '-k', f'{port}/tcp'], capture_output=True, text=True)
        killed = (out.returncode == 0)
    except FileNotFoundError:
        out = subprocess.run(
            ['bash', '-c', f"lsof -ti tcp:{port} | xargs -r kill -9"],
            capture_output=True, text=True,
        )
        killed = bool(out.stdout.strip())
    if killed:
        print(f'killed prior process on :{port}')
        time.sleep(1)

_kill_port(PORT)

if not _env_true('COLAB_MOCK_MODEL'):
    try:
        import torch
        cuda_ok = bool(torch.cuda.is_available())
    except Exception:
        cuda_ok = False
    if not cuda_ok:
        raise RuntimeError(
            'No CUDA GPU detected, but COLAB_MOCK_MODEL is false. '
            'Attach a GPU runtime (Runtime -> Change runtime type -> GPU) '
            'or set os.environ["COLAB_MOCK_MODEL"] = "true" before launching.'
        )

STDOUT = STDOUT_PATH.open('ab', buffering=0)
STDERR = STDERR_PATH.open('ab', buffering=0)
for handle, name in ((STDOUT, 'stdout'), (STDERR, 'stderr')):
    handle.write(f'\n--- launch {time.strftime("%Y-%m-%d %H:%M:%S")} {name} ---\n'.encode())

env = os.environ.copy()
env['PYTHONPATH'] = REPO_DIR + (':' + env['PYTHONPATH'] if env.get('PYTHONPATH') else '')

proc = subprocess.Popen(
    [
        'python', '-m', 'uvicorn',
        'colab.text_to_sql_colab_server:app',
        '--host', '0.0.0.0',
        '--port', str(PORT),
        '--log-level', 'info',
    ],
    cwd=REPO_DIR,
    env=env,
    stdout=STDOUT, stderr=STDERR,
)
print('uvicorn pid:', proc.pid)
print('stdout log:', STDOUT_PATH)
print('stderr log:', STDERR_PATH)

URL = f'http://127.0.0.1:{PORT}/health'
deadline = time.time() + 600
last_err = None
last_print = 0
while time.time() < deadline:
    rc = proc.poll()
    if rc is not None:
        raise RuntimeError(
            f'uvicorn exited early with code {rc}.\n\n'
            f'--- stderr tail ---\n{_tail(STDERR_PATH)}\n\n'
            f'--- stdout tail ---\n{_tail(STDOUT_PATH)}'
        )
    try:
        with request.urlopen(URL, timeout=5) as resp:
            payload = _json.loads(resp.read())
            if payload.get('model_loaded') or payload.get('mock_model'):
                print('health ok:')
                print(_json.dumps(payload, ensure_ascii=False, indent=2))
                break
            if time.time() - last_print >= 15:
                print('waiting for model_loaded ... status:')
                print(_json.dumps(payload, ensure_ascii=False, indent=2))
                last_print = time.time()
    except (error.URLError, TimeoutError, ConnectionError) as exc:
        last_err = exc
        if time.time() - last_print >= 15:
            print('waiting for /health ... last error:', repr(exc))
            print('stderr tail:')
            print(_tail(STDERR_PATH, lines=20))
            last_print = time.time()
    time.sleep(5)
else:
    raise RuntimeError(
        f'service did not become healthy in time: {last_err}\n\n'
        f'--- stderr tail ---\n{_tail(STDERR_PATH)}\n\n'
        f'--- stdout tail ---\n{_tail(STDOUT_PATH)}'
    )


uvicorn pid: 2833
stdout log: /content/drive/MyDrive/nl2bi_colab/logs/uvicorn.stdout.log
stderr log: /content/drive/MyDrive/nl2bi_colab/logs/uvicorn.stderr.log
waiting for /health ... last error: URLError(ConnectionRefusedError(111, 'Connection refused'))
stderr tail:
Loading weights: 100%|██████████| 339/339 [00:01<00:00, 313.76it/s, Materializing param=model.norm.weight]
2026-05-11 18:56:08,621 INFO httpx HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-Coder-7B-Instruct/resolve/main/generation_config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-11 18:56:08,633 INFO httpx HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-Coder-7B-Instruct/c03e6d358207e414f1eca0bb1891e29f1db0e242/generation_config.json "HTTP/1.1 200 OK"
2026-05-11 18:56:08,739 INFO httpx HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-Coder-7B-Instruct/resolve/main/custom_generate/generate.py "HTTP/1.1 404 Not Found"

--- launch 2026-05-11 19:08:55 stderr ---
waiting for /h

In [9]:
# 8. Open ngrok tunnel. Token comes from env, never printed.
import os
from pyngrok import ngrok, conf

tok = os.environ.get('NGROK_AUTHTOKEN')
if not tok:
    raise RuntimeError('NGROK_AUTHTOKEN missing. Add it to /content/drive/MyDrive/nl2bi_colab/.env')
conf.get_default().auth_token = tok

for tunnel in list(ngrok.get_tunnels()):
    ngrok.disconnect(tunnel.public_url)

PORT = int(os.environ.get('COLAB_PORT', '8000'))
tunnel = ngrok.connect(PORT, 'http')
PUBLIC_URL = tunnel.public_url.replace('http://', 'https://')
public_url_marker = Path(os.environ.get('COLAB_PUBLIC_URL_MARKER', '/content/drive/MyDrive/nl2bi_colab/.public_url'))
public_url_marker.parent.mkdir(parents=True, exist_ok=True)
public_url_marker.write_text(PUBLIC_URL + '\n', encoding='utf-8')
print('PUBLIC_URL:', PUBLIC_URL)
print('PUBLIC_URL_MARKER:', public_url_marker)

PUBLIC_URL: https://791e-34-13-156-2.ngrok-free.app                                                 


In [10]:
# 8b. Start the agent bridge — Flask /exec inside this kernel, ngrok tunnel.
# Lets the agent push code into Colab without you clicking "Run cell" later.
# After this cell runs, the agent can fetch the bridge URL from
# {PUBLIC_URL}/admin/bridge_url, then POST to /exec for git pull, uvicorn
# restart, etc. Idempotent: re-runs reuse the same flask thread.
import sys, importlib
if '/content/nl2bi-colab' not in sys.path:
    sys.path.insert(0, '/content/nl2bi-colab')
import colab.agent_bridge as agent_bridge
importlib.reload(agent_bridge)  # in case we git pulled a newer agent_bridge.py
BRIDGE_URL = agent_bridge.start_bridge()
print('BRIDGE_URL (also at', f'{PUBLIC_URL}/admin/bridge_url):', BRIDGE_URL)


 * Serving Flask app 'colab.agent_bridge'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5050
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [11/May/2026 19:10:05] "GET /health HTTP/1.1" 200 -


BRIDGE: flask started on 127.0.0.1:5050
BRIDGE: wrote URL marker to /content/drive/MyDrive/nl2bi_colab/.bridge_url

BRIDGE_URL: https://3e40-34-13-156-2.ngrok-free.app
BRIDGE_AUTH: X-Bridge-Token header required on all routes except /health
BRIDGE_READY
BRIDGE_URL (also at https://791e-34-13-156-2.ngrok-free.app/admin/bridge_url): https://3e40-34-13-156-2.ngrok-free.app


In [11]:
# 9. Smoke-test against the public tunnel. Captures and prints stdout so output survives in the cell.
import subprocess, sys
res = subprocess.run(
    [sys.executable, '-m', 'colab.smoke_extract', '--base-url', PUBLIC_URL],
    cwd='/content/nl2bi-colab',
    capture_output=True, text=True,
)
print(res.stdout)
if res.stderr:
    print('--- stderr ---')
    print(res.stderr)
print(f'(smoke exit code: {res.returncode})')


== /health @ https://791e-34-13-156-2.ngrok-free.app ==
http 200
{
  "status": "ok",
  "model_loaded": true,
  "model_id": "Qwen/Qwen2.5-Coder-7B-Instruct",
  "mock_model": false,
  "planner_enabled": false,
  "planner_loaded": false,
  "planner_id": null,
  "device": "cuda",
  "gpu_name": "NVIDIA RTX PRO 6000 Blackwell Server Edition",
  "vram_total_gb": 94.97,
  "vram_free_gb": 79.63,
  "demo_db_ready": true,
  "server_role": "colab-runtime",
  "auth": {
    "require_auth": true,
    "api_token_set": true,
    "debug_endpoints": false,
    "bridge_enabled": false
  }
}

== /extract @ category_comparison.json ==
http 200 (round-trip 3268 ms)
{
  "request_id": "smoke-category-comparison",
  "status": "success",
  "sql": "SELECT singer.Country, COUNT(singer.Singer_ID) AS NumberOfSingers\nFROM singer\nGROUP BY singer.Country\nLIMIT 1000",
  "row_count": 3,
  "truncated": false,
  "columns": [
    "Country",
    "NumberOfSingers"
  ],
  "errors": [],
  "warnings": [],
  "latency_ms": 2945

In [12]:
# 10. (Optional) Stop server + tunnel before disconnecting runtime.
# from pyngrok import ngrok
# for t in list(ngrok.get_tunnels()):
#     ngrok.disconnect(t.public_url)
# proc.terminate(); proc.wait(timeout=10)
# print('stopped')
pass

INFO:werkzeug:127.0.0.1 - - [11/May/2026 19:11:31] "GET /health HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [11/May/2026 19:11:32] "POST /exec HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [11/May/2026 19:11:36] "POST /exec HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [11/May/2026 19:13:51] "POST /exec HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [11/May/2026 19:14:03] "POST /exec HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [11/May/2026 19:14:09] "POST /exec HTTP/1.1" 200 -
